[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_proteomics.ipynb) [![View in nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_proteomics.ipynb)

# Proteomics

Run PAC, HPS, chronological and mortality OrganAge, and PAOPAC using **deterministic synthetic data**. The values demonstrate the API and input preparation; they are not participant measurements or evidence of biological performance. Install `pyaging` before running this notebook. Model assets download from Hugging Face on first use.

These models were developed with human plasma measured by **Olink Explore 3072**. Supply assay-normalized NPX, a log2 relative-abundance measure. Negative NPX can be valid. Protein names alone do not establish equivalence between assays, panels, batches, serum and plasma. pyaging does not convert raw assay counts to NPX or automatically harmonize platforms.

| Clock | Output |
| --- | --- |
| PAC | Mortality-equivalent age in years |
| HPS | Probability of avoiding the study's major-condition-or-death endpoint over ten years |
| OrganAge chronological | Predicted age in years |
| OrganAge mortality | Relative natural-log mortality hazard |
| PAOPAC Conventional | Age in years after correction fitted to the submitted cohort |

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import pyaging as pya

clock_names = [
    "pac", "hps",
    "organagechronologicalconventional", "organagemortalityconventional",
    "paopac",
]
feature_tables = {name: pya.utils.get_feature_ranges(name) for name in clock_names}
features = {name: table["feature"].tolist() for name, table in feature_tables.items()}
pd.DataFrame({"input_columns": {name: len(names) for name, names in features.items()}})

,input_columns
pac,129
hps,87
organagechronologicalconventional,2916
organagemortalityconventional,2916
paopac,191


## Build aligned inputs

Use the exact assay identifiers from each model's feature table. PAC/HPS use lowercase names, OrganAge uses case-sensitive symbols such as `NTproBNP` and `HLA-DRA`, and PAOPAC uppercases protein names internally. The separate matrices below demonstrate these interfaces; they do not provide a cross-platform mapping.

The same 32 synthetic sample identifiers and ages are used throughout. The helper aligns age by identifier and checks that every sample has a finite age. Constructing `AnnData` directly preserves NaNs. `pya.pp.df_to_adata` performs imputation, so use it only when that preparation is deliberate.

In [2]:
rng = np.random.default_rng(20261002)
samples = pd.Index([f"synthetic_{i:02d}" for i in range(32)], name="sample")
age = pd.Series(np.linspace(40.0, 70.0, len(samples)), index=samples, name="age")

def synthetic_npx(columns):
    return pd.DataFrame(
        rng.normal(0.0, 0.7, size=(len(samples), len(columns))),
        index=samples,
        columns=columns,
    )

def with_age(npx, age_years):
    if not npx.index.is_unique or not age_years.index.is_unique:
        raise ValueError("Sample identifiers must be unique")
    if not npx.columns.is_unique or "age" in npx.columns:
        raise ValueError("Resolve duplicate or reserved input columns")
    aligned = age_years.reindex(npx.index).astype(float)
    if not np.isfinite(aligned).all():
        raise ValueError("Every sample needs a finite chronological age in years")
    return npx.assign(age=aligned)

# The table reports units, not evidence of compatibility with a different assay.
feature_tables["pac"].head()

,feature,unit,low,high
0,age,years,0.0,122.5
1,ada2,NPX,-inf,inf
2,adamts13,NPX,-inf,inf
3,adamts16,NPX,-inf,inf
4,adgrg2,NPX,-inf,inf


## PAC and HPS

[PAC](https://github.com/kuo-lab-uchc/PAC) requires 128 proteins plus `age` in years. [HPS](https://github.com/kuo-lab-uchc/HPS) requires 86 proteins plus `age`. Keep the original lowercase assay names, including `fut3_fut5`. HPS requires both `nppb` and `ntprobnp`; do not merge them.

Supply the assay's normalized NPX without additional centering, exponentiation or scaling. All required columns must be present; absent columns raise an error, and supplied NaNs propagate. Any upstream imputation is a separate choice. The HPS study used within-batch and across-batch intensity normalization and upstream k-nearest-neighbor imputation with `k=10`.

HPS returns a probability from 0 to 1, with higher values healthier. Its endpoint covers cancer excluding nonmelanoma skin cancer, diabetes, heart failure, myocardial infarction, stroke, COPD, dementia and death. It does not return years of healthy life. See the [HPS paper](https://doi.org/10.1073/pnas.2414086122).

In [3]:
kuo_proteins = sorted((set(features["pac"]) | set(features["hps"])) - {"age"})
kuo_npx = synthetic_npx(kuo_proteins)
kuo_inputs = with_age(kuo_npx, age)
for name in ["pac", "hps"]:
    missing = set(features[name]) - set(kuo_inputs.columns)
    if missing:
        raise ValueError(f"{name}: missing required inputs {sorted(missing)}")

kuo_adata = ad.AnnData(kuo_inputs)
pya.pred.predict_age(kuo_adata, ["pac", "hps"], verbose=False)
assert np.isfinite(kuo_adata.obs[["pac", "hps"]]).all().all()
assert kuo_adata.obs["hps"].between(0, 1).all()
kuo_adata.obs[["pac", "hps"]].head()

,pac,hps
sample,,
synthetic_00,50.940487,0.906109
synthetic_01,48.538009,0.930831
synthetic_02,31.523923,0.920229
synthetic_03,58.111728,0.824064
synthetic_04,43.577574,0.944730


## OrganAge

The 46 current OrganAge clocks use the authors' recommended **first fold of the full Explore 3072 models**. Each family has 23 outputs. Here, `conventional` uses proteins across organs. For an organ-specific output, select a name such as `organagechronologicalbrain`; these models use circulating proteins associated with the organ, not organ biopsies. Female and Male variants do not require a sex input.

Keep the exact protein symbols and NPX scale. pyaging applies the coefficient sum and chronological intercept without further scaling. Missing columns contribute zero, matching omission in the author's example; this is not reference-mean imputation. Present NaNs propagate. Inspect coverage before interpreting any score.

For other assays, the [authors' preparation guidance](https://github.com/ludgergoeminne/organAging#3-other-data-types) describes making distributions approximately symmetric, subtracting protein-wise means, dividing by protein-wise SDs and multiplying by the **full-model UKB SDs in Table S3**. The optional SD-matching block in `Example_Script.R` omits mean subtraction. Neither procedure runs automatically in pyaging. Record the chosen procedure and reference cohort; rescaling a new cohort can change predictions. See the [OrganAge paper](https://doi.org/10.1016/j.cmet.2024.10.005).

In [4]:
organ_clocks = ["organagechronologicalconventional", "organagemortalityconventional"]
organ_proteins = sorted(set().union(*(features[name] for name in organ_clocks)))
organ_npx = synthetic_npx(organ_proteins)
organ_adata = ad.AnnData(organ_npx)
pya.pred.predict_age(organ_adata, organ_clocks, verbose=False)
assert np.isfinite(organ_adata.obs[organ_clocks]).all().all()
organ_adata.obs[organ_clocks].head()

,organagechronologicalconventional,organagemortalityconventional
sample,,
synthetic_00,61.203524,-0.035365
synthetic_01,52.591469,-0.304399
synthetic_02,61.347902,-0.249981
synthetic_03,51.681665,0.232191
synthetic_04,64.548275,-0.050437


## PAOPAC with the full cohort

`paopac` reproduces the Conventional output of the [authors' public v1.0.0 release](https://github.com/JackieHanLab/PAOPAC/releases/tag/v1.0.0). It uses 189 proteins, optional matrix `TDI`, and required `age` in years. Supply NPX directly. Internally, it averages exact duplicate protein names on the NPX scale, uppercases names, applies `2**NPX`, fills missing or absent linear abundances with zero, and fits `StandardScaler` across the complete submitted cohort. Names that collide only after uppercasing are rejected.

After the 800-tree prediction, the model subtracts a LOWESS fit of predicted-minus-chronological age against chronological age, using `frac=0.75` and the original statsmodels defaults. These steps were verified against the authors' compiled interface; they are not inferred from protein-name overlap.

`TDI` is normally absent. If supplied, it must be a matrix column and undergoes the same exponentiation as proteins. `TDI` in `.obs` is ignored; this is not a conventional adjustment for a raw Townsend deprivation score. Keep NPX NaNs if you intend the author's linear-scale zero fill: filling NPX with zero first gives `2**0 = 1` instead.

Pass the full analysis cohort in one call. `batch_size` controls computation, not the normalization cohort. The example omits optional `TDI` and includes one NaN to exercise the original missing-value rule.

In [5]:
paopac_proteins = [name for name in features["paopac"] if name not in {"age", "TDI"}]
paopac_npx = synthetic_npx(paopac_proteins)
paopac_npx.iloc[0, 0] = np.nan
paopac_adata = ad.AnnData(with_age(paopac_npx, age))
pya.pred.predict_age(paopac_adata, "paopac", batch_size=7, verbose=False)
assert np.isfinite(paopac_adata.obs["paopac"]).all()
assert np.isnan(paopac_adata.X[0, 0])  # input NaNs remain unchanged
paopac_adata.obs["paopac"].head()

sample
synthetic_00    36.037396
synthetic_01    41.096683
synthetic_02    44.507700
synthetic_03    47.468443
synthetic_04    45.039386
Name: paopac, dtype: float64

Changing cohort membership or its ages changes PAOPAC results. A singleton returns its supplied chronological age, and all-missing protein cohorts can also collapse to chronological age. Equal-age LOWESS ties can depend on row order. These behaviors do not validate individual scoring or comparisons between separately formed tiny cohorts. The output is not a portable age-acceleration residual.

## Inspect coverage and units

`percent_na` reports absent model features, not every NaN in supplied columns. Check both. Here all PAC/HPS and OrganAge inputs are present; PAOPAC has only optional `TDI` absent and one supplied NPX NaN. Missing influential proteins can still alter scores when a model permits them.

The following table labels the output units explicitly. The synthetic values should not be used to compare clock performance.

In [6]:
outputs = {
    "pac": (kuo_adata, "mortality-equivalent years"),
    "hps": (kuo_adata, "10-year endpoint-free probability"),
    organ_clocks[0]: (organ_adata, "years"),
    organ_clocks[1]: (organ_adata, "relative natural-log mortality hazard"),
    "paopac": (paopac_adata, "cohort-corrected years"),
}
coverage = pd.DataFrame([
    {
        "clock": name,
        "output_unit": unit,
        "absent_features_percent": obj.uns[f"{name}_percent_na"],
        "absent_features": obj.uns[f"{name}_missing_features"],
    }
    for name, (obj, unit) in outputs.items()
]).set_index("clock")
coverage

,output_unit,absent_features_percent,absent_features
clock,,,
pac,mortality-equivalent years,0.00000,[]
hps,10-year endpoint-free probability,0.00000,[]
organagechronologicalconventional,years,0.00000,[]
organagemortalityconventional,relative natural-log mortality hazard,0.00000,[]
paopac,cohort-corrected years,0.52356,[TDI]


In [7]:
pd.Series({
    "PAC/HPS supplied NPX NaNs": int(kuo_npx.isna().sum().sum()),
    "OrganAge supplied NPX NaNs": int(organ_npx.isna().sum().sum()),
    "PAOPAC supplied NPX NaNs": int(paopac_npx.isna().sum().sum()),
})

PAC/HPS supplied NPX NaNs     0
OrganAge supplied NPX NaNs    0
PAOPAC supplied NPX NaNs      1
dtype: int64

Citations and clock-specific notes are stored in `adata.uns["<clock_name>_metadata"]`. The [source follow-up](https://github.com/lucascamillomd/pyaging/blob/main/docs/development/proteomic-source-follow-up.md) records original-author comparisons, unavailable ProtAge/ipfP3GPT assets, and the unverified equivalence between public PAOPAC and the private trial version. Those limitations prevent treating these examples as a reproduction of the six-clock trial.

Consult each model's metadata and original terms before reuse.

In [8]:
{
    name: obj.uns[f"{name}_metadata"]["doi"]
    for name, (obj, _) in outputs.items()
}

{'pac': 'https://doi.org/10.1111/acel.14195',
 'hps': 'https://doi.org/10.1073/pnas.2414086122',
 'organagechronologicalconventional': 'https://doi.org/10.1016/j.cmet.2024.10.005',
 'organagemortalityconventional': 'https://doi.org/10.1016/j.cmet.2024.10.005',
 'paopac': 'https://doi.org/10.64898/2026.04.24.720503'}